# Carnaval / Recife — Instagram Epidemiological Scraper (local LLM)

End-to-end pipeline running **100% locally** on Ollama (no cloud API, no keys):

1. **Login once** → save a browser session.
2. **Collect links** from hashtags *and/or* select pages.
3. **Scrape posts** → extract `username / text / location` with a local LLM (qwen2.5).
4. **Scrape stories** from a list of select pages.
5. **Classify** each item for public-health risk.

All heavy logic lives in `ig_scraper.py`; this notebook is just the driver.

---
### Prerequisites (run on the Windows / RTX 4000 Ada box)
```powershell
pip install -r requirements.txt
python -m playwright install chromium

ollama pull qwen2.5:14b        # extraction + classification (~9 GB)
ollama pull nomic-embed-text   # embeddings for ScrapeGraphAI (~0.3 GB)
# optional faster model for big batches:
# ollama pull qwen2.5:7b
```
Model choice and the reasoning for the 20 GB GPU are in **README.md**.

In [ ]:
# Lets the async functions run inside Jupyter's already-running event loop.
import nest_asyncio; nest_asyncio.apply()

import importlib, config, ig_scraper
importlib.reload(config); importlib.reload(ig_scraper)
print("Extraction model:", config.EXTRACTION_MODEL)
print("Classifier model:", config.CLASSIFIER_MODEL)
print("Embeddings model:", config.EMBEDDINGS_MODEL)

## 1. Login (run once) — saves `ig_state.json`

In [ ]:
# A real browser opens; log in by hand, then the session is saved and reused.
await ig_scraper.login_and_save_state("ig_state.json", wait_seconds=120)

## 2a. Collect post links from hashtags

In [ ]:
links_by_hashtag, post_links = await ig_scraper.collect_links_for_hashtags(
    config.HASHTAGS,                 # ["carnaval", "carnavalrecife", "recife"]
    max_links_per_hashtag=200,
    scrolls=50,
)
len(post_links), post_links[:10]

## 2b. (Optional) Collect post links from select pages

In [ ]:
page_links = await ig_scraper.collect_links_for_pages(
    config.FEED_PAGES,               # e.g. ["prefeituradorecife", "saude_recife"]
    max_links_per_page=60,
)
len(page_links), page_links[:10]

## 3. Scrape posts → local-LLM extraction → CSV

Each post's logged-in HTML is parsed by `qwen2.5` via ScrapeGraphAI into
`username / text / location`.

In [ ]:
df_posts = await ig_scraper.scrape_posts(
    post_links,                      # or: page_links
    out_csv=config.POSTS_CSV,
    max_n=None,
    concurrency=2,
)
df_posts.head()

## 4. Scrape STORIES from a list of select pages

For every account in `config.STORY_PAGES` the story viewer is opened and each
segment is captured: **timestamp, media URL, overlay/DOM text, and a screenshot**
(saved to `stories_media/`). Stories are ephemeral — only currently-live stories
are returned.

In [ ]:
# Edit the list of pages here or in config.STORY_PAGES.
story_pages = config.STORY_PAGES   # ["prefeituradorecife", "saude_recife", "carnaval"]

df_stories = await ig_scraper.scrape_stories_for_pages(
    story_pages,
    out_csv=config.STORIES_CSV,
    max_segments_per_user=40,
)
df_stories.head(20)

## 5. Public-health risk classification

In [ ]:
# Classify the scraped POSTS. (Point in_path at STORIES_CSV + text_col='overlay_text'
# to classify story text instead.)
df_risk = ig_scraper.classify_risk_csv(
    in_path=config.POSTS_CSV,
    out_path=config.RISK_CSV,
    text_col="text",
    loc_col="location",
)
df_risk[["username", "location", "risk", "risk_signal"]].head(20)

## (Optional) Classify story overlay text

In [ ]:
df_story_risk = ig_scraper.classify_risk_csv(
    in_path=config.STORIES_CSV,
    out_path="carnaval_stories_with_risk.csv",
    text_col="overlay_text",
    loc_col="username",
)
df_story_risk.head(20)